In [2]:
from __future__ import annotations

import argparse
from pathlib import Path
from typing import Dict, Mapping, Sequence, Tuple

import numpy as np
import torch
from torch import Tensor
from torch.utils.data import DataLoader, Dataset

try:
    from .model import RITCFModel, build_model
    from .train import apply_normalization, channel_first, find_npy, metrics_from_counts
except ImportError:
    from model import RITCFModel, build_model
    from train import apply_normalization, channel_first, find_npy, metrics_from_counts

"""
python evaluate.py \
  --data-dir .\code1\data \
  --checkpoint .\checkpoints\best_model.pt \
  --votes 10 \
  --min-ri-votes 6 \
  --batch-size 256

"""

ArrayDict = Dict[str, np.ndarray]
TensorDict = Dict[str, Tensor]


class TestPairDataset(Dataset):
    def __init__(self, input_a: ArrayDict, input_b: ArrayDict) -> None:
        self.input_a = input_a
        self.input_b = input_b
        self.length = len(next(iter(input_a.values())))

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, idx: int) -> Tuple[TensorDict, TensorDict]:
        a = {key: torch.from_numpy(value[idx]).float() for key, value in self.input_a.items()}
        b = {key: torch.from_numpy(value[idx]).float() for key, value in self.input_b.items()}
        return a, b


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default=".")
    parser.add_argument("--checkpoint", required=True)
    parser.add_argument("--batch-size", type=int, default=256)
    parser.add_argument("--num-workers", type=int, default=0)
    parser.add_argument("--threshold", type=float, default=None)
    parser.add_argument("--votes", type=int, default=10)
    parser.add_argument("--min-ri-votes", type=int, default=6)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    return parser.parse_args()


def load_test_split(data_dir: Path, split: str, checkpoint: Mapping) -> Tuple[ArrayDict, ArrayDict, np.ndarray]:
    input_a: ArrayDict = {}
    input_b: ArrayDict = {}
    for name in checkpoint["field20"]:
        input_a[name] = channel_first(np.load(find_npy(data_dir, split, name, "A")), 20).astype(np.float32)
        input_b[name] = channel_first(np.load(find_npy(data_dir, split, name, "B")), 20).astype(np.float32)
    for name in checkpoint["field5"]:
        input_a[name] = channel_first(np.load(find_npy(data_dir, split, name, "A")), 5).astype(np.float32)
        input_b[name] = channel_first(np.load(find_npy(data_dir, split, name, "B")), 5).astype(np.float32)
    for name in checkpoint["satellite"]:
        input_a[name] = channel_first(np.load(find_npy(data_dir, split, name, "A")), 5).astype(np.float32)
        input_b[name] = channel_first(np.load(find_npy(data_dir, split, name, "B")), 5).astype(np.float32)
    if checkpoint["use_his"]:
        input_a["his"] = np.load(find_npy(data_dir, split, "his", "A")).astype(np.float32)
        input_b["his"] = np.load(find_npy(data_dir, split, "his", "B")).astype(np.float32)

    try:
        label_path = find_npy(data_dir, split, "label", None)
    except FileNotFoundError:
        label_path = find_npy(data_dir, split, "y", None)
    label = np.load(label_path).astype(np.float32).reshape(-1)
    return input_a, input_b, label


def flatten_votes(input_a: ArrayDict, input_b: ArrayDict, label: np.ndarray, votes: int) -> Tuple[ArrayDict, ArrayDict, np.ndarray, int]:
    first = next(iter(input_a.values()))
    if first.ndim >= 5 and first.shape[1] == votes:
        event_count = first.shape[0]
        flat_a = {key: value.reshape((event_count * votes,) + value.shape[2:]) for key, value in input_a.items()}
        flat_b = {key: value.reshape((event_count * votes,) + value.shape[2:]) for key, value in input_b.items()}
        event_label = label.reshape(event_count, -1)[:, 0] if label.size != event_count else label
        return flat_a, flat_b, event_label, event_count

    if len(first) % votes != 0:
        raise ValueError(f"test pair count {len(first)} is not divisible by votes={votes}")
    event_count = len(first) // votes
    event_label = label.reshape(event_count, -1)[:, 0] if label.size != event_count else label
    return input_a, input_b, event_label, event_count


def to_device(batch: Mapping[str, Tensor], device: torch.device) -> TensorDict:
    return {key: value.to(device, non_blocking=True) for key, value in batch.items()}


@torch.no_grad()
def predict_distances(model: RITCFModel, loader: DataLoader, device: torch.device) -> np.ndarray:
    model.eval()
    distances = []
    for input_a, input_b in loader:
        input_a = to_device(input_a, device)
        input_b = to_device(input_b, device)
        distances.append(model(input_a, input_b).detach().cpu().numpy().reshape(-1))
    return np.concatenate(distances, axis=0)


def voting_metrics(distance: np.ndarray, label: np.ndarray, threshold: float, votes: int, min_ri_votes: int) -> Dict[str, float]:
    distance = distance.reshape(-1, votes)
    ri_votes = np.sum(distance < threshold, axis=1)
    pred_ri = ri_votes >= min_ri_votes
    true_ri = label.reshape(-1) < 0.5

    hit = int(np.sum(true_ri & pred_ri))
    false_alarm = int(np.sum(~true_ri & pred_ri))
    miss = int(np.sum(true_ri & ~pred_ri))
    correct_negative = int(np.sum(~true_ri & ~pred_ri))
    metrics = metrics_from_counts(hit, false_alarm, miss, correct_negative)
    metrics.update(
        {
            "hit": hit,
            "false_alarm": false_alarm,
            "miss": miss,
            "correct_negative": correct_negative,
            "event_count": int(len(label)),
            "mean_ri_votes": float(np.mean(ri_votes)),
        }
    )
    return metrics


def main() -> None:
    args = parse_args()
    checkpoint = torch.load(args.checkpoint, map_location="cpu")
    threshold = float(checkpoint.get("threshold", 0.5) if args.threshold is None else args.threshold)

    input_a, input_b, label = load_test_split(Path(args.data_dir), "test", checkpoint)
    split = (input_a, input_b, label)
    apply_normalization(split, checkpoint["normalization"])
    input_a, input_b, label, event_count = flatten_votes(input_a, input_b, label, args.votes)

    dataset = TestPairDataset(input_a, input_b)
    loader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=args.device.startswith("cuda"),
    )

    device = torch.device(args.device)
    model = build_model(
        field20_names=checkpoint["field20"],
        field5_names=checkpoint["field5"],
        satellite_names=checkpoint["satellite"],
        use_his=checkpoint["use_his"],
        his_dim=checkpoint["his_dim"],
    ).to(device)
    model.load_state_dict(checkpoint["model"])

    distance = predict_distances(model, loader, device)
    metrics = voting_metrics(distance, label, threshold, args.votes, args.min_ri_votes)

    print(
        f"test_events={event_count} "
        f"POD={metrics['POD']:.4f} "
        f"FARate={metrics['FARate']:.4f} "
        f"hit={metrics['hit']} "
        f"false_alarm={metrics['false_alarm']} "
        f"miss={metrics['miss']} "
        f"correct_negative={metrics['correct_negative']} "
        f"mean_RI_votes={metrics['mean_ri_votes']:.4f}"
    )


if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'torch'

In [3]:
from __future__ import annotations

import argparse
from typing import Dict, List, Mapping, Optional, Sequence

import torch
from torch import Tensor, nn


class ConvStage(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class FactorBranch(nn.Module):
    def __init__(self, in_channels: int, descriptor_dim: int = 15, dropout: float = 0.0) -> None:
        super().__init__()
        self.block1 = ConvStage(in_channels, 128)
        self.block2 = ConvStage(128, 256)
        self.block3 = ConvStage(256, 512)

        head: List[nn.Module] = [nn.Flatten()]
        if dropout > 0:
            head.append(nn.Dropout(dropout))
        head.append(nn.Linear(512 * 4 * 4, descriptor_dim))
        self.head = nn.Sequential(*head)

    def forward_features(self, x: Tensor) -> List[Tensor]:
        f1 = self.block1(x)
        f2 = self.block2(f1)
        f3 = self.block3(f2)
        return [f1, f2, f3]

    def forward_descriptor(self, features: Sequence[Tensor]) -> Tensor:
        return self.head(features[-1])


class AddBranch(nn.Module):
    def __init__(self, descriptor_dim: int = 15, dropout: float = 0.0) -> None:
        super().__init__()
        self.block2 = ConvStage(128, 256)
        self.block3 = ConvStage(256, 512)

        head: List[nn.Module] = [nn.Flatten()]
        if dropout > 0:
            head.append(nn.Dropout(dropout))
        head.append(nn.Linear(512 * 4 * 4, descriptor_dim))
        self.head = nn.Sequential(*head)

    def forward(self, multi_factor_features: Sequence[Sequence[Tensor]]) -> Tensor:
        level1 = torch.stack([features[0] for features in multi_factor_features], dim=0).sum(dim=0)
        level2 = self.block2(level1)
        level2 = level2 + torch.stack([features[1] for features in multi_factor_features], dim=0).sum(dim=0)
        level3 = self.block3(level2)
        level3 = level3 + torch.stack([features[2] for features in multi_factor_features], dim=0).sum(dim=0)
        return self.head(level3)


class AllBranch(nn.Module):
    def __init__(self, in_channels: int, descriptor_dim: int = 15, dropout: float = 0.0) -> None:
        super().__init__()
        self.block3 = ConvStage(in_channels, 128)
        self.block4a = ConvStage(128, 256)
        self.block4b = ConvStage(256, 512)

        head: List[nn.Module] = [nn.Flatten()]
        if dropout > 0:
            head.append(nn.Dropout(dropout))
        head.append(nn.Linear(512 * 4 * 4, descriptor_dim))
        self.head = nn.Sequential(*head)

    def forward(self, x: Tensor) -> Tensor:
        x = self.block3(x)
        x = self.block4a(x)
        x = self.block4b(x)
        return self.head(x)


class SatelliteBranch(nn.Module):
    def __init__(self, descriptor_dim: int = 15, dropout: float = 0.0) -> None:
        super().__init__()
        self.net = nn.Sequential(
            ConvStage(5, 32),
            ConvStage(32, 64),
            ConvStage(64, 128),
            ConvStage(128, 256),
            ConvStage(256, 512),
        )
        head: List[nn.Module] = [nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten()]
        if dropout > 0:
            head.append(nn.Dropout(dropout))
        head.append(nn.Linear(512 * 4 * 4, descriptor_dim))
        self.head = nn.Sequential(*head)

    def forward(self, x: Tensor) -> Tensor:
        return self.head(self.net(x))


class HisBranch(nn.Module):
    def __init__(self, in_dim: int, descriptor_dim: int = 15, dropout: float = 0.0) -> None:
        super().__init__()
        layers: List[nn.Module] = [
            nn.Linear(in_dim, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 64),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        layers.append(nn.Linear(64, descriptor_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class ResidualHead(nn.Module):
    def __init__(self, in_dim: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 256)
        self.fc2 = nn.Linear(256, 15)
        self.fc3 = nn.Linear(in_dim + 15, 90)
        self.fc4 = nn.Linear(90, 256)
        self.fc5 = nn.Linear(256, 256)
        self.fc6 = nn.Linear(256, 256)
        self.fc7 = nn.Linear(256, 128)
        self.fc8 = nn.Linear(128, 128)
        self.fc9 = nn.Linear(128, 128)
        self.output = nn.Linear(128, 1)
        self.activation = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: Tensor) -> Tensor:
        shortcut_input = x
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = torch.cat([x, shortcut_input], dim=1)
        x = self.activation(self.fc3(x))
        x = self.activation(self.fc4(x))

        residual = x
        x = self.activation(self.fc5(x))
        x = self.dropout(self.activation(self.fc6(x)))
        x = self.activation(x + residual)

        x = self.activation(self.fc7(x))
        residual = x
        x = self.activation(self.fc8(x))
        x = self.dropout(self.activation(self.fc9(x)))
        x = self.activation(x + residual)
        return self.output(x)


class SampleEncoder(nn.Module):
    def __init__(
        self,
        field20_names: Sequence[str],
        field5_names: Sequence[str],
        satellite_names: Sequence[str],
        use_his: bool,
        his_dim: int = 30,
        descriptor_dim: int = 15,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.field20_names = list(field20_names)
        self.field5_names = list(field5_names)
        self.satellite_names = list(satellite_names)
        self.use_his = use_his
        self.grid_names = self.field20_names + self.field5_names
        self.grid_channels = {name: 20 for name in self.field20_names}
        self.grid_channels.update({name: 5 for name in self.field5_names})

        if not self.field20_names:
            raise ValueError("at least one 20x25x25 factor is required")
        if not self.grid_names and not self.satellite_names and not use_his:
            raise ValueError("at least one input group must be enabled")

        self.factor_branches = nn.ModuleDict(
            {name: FactorBranch(20, descriptor_dim=descriptor_dim, dropout=dropout) for name in self.field20_names}
        )
        self.add_branch = AddBranch(descriptor_dim=descriptor_dim, dropout=dropout)
        self.all_branch = AllBranch(
            in_channels=sum(self.grid_channels.values()),
            descriptor_dim=descriptor_dim,
            dropout=dropout,
        )
        self.satellite_branches = nn.ModuleDict(
            {name: SatelliteBranch(descriptor_dim=descriptor_dim, dropout=dropout) for name in self.satellite_names}
        )
        self.his_branch: Optional[HisBranch] = (
            HisBranch(his_dim, descriptor_dim=descriptor_dim, dropout=dropout) if use_his else None
        )

        descriptor_count = len(self.field20_names) + 2 + len(self.satellite_names) + int(use_his)
        self.descriptor_dim = descriptor_count * descriptor_dim
        self.output_dim = 1
        self.head = ResidualHead(self.descriptor_dim, dropout=dropout)

    def forward(self, x: Mapping[str, Tensor]) -> Tensor:
        missing = [name for name in self.required_inputs if name not in x]
        if missing:
            raise KeyError(f"missing model inputs: {missing}")

        descriptors: List[Tensor] = []
        multi_factor_features: List[List[Tensor]] = []
        for name in self.field20_names:
            features = self.factor_branches[name].forward_features(x[name])
            multi_factor_features.append(features)
            descriptors.append(self.factor_branches[name].forward_descriptor(features))

        descriptors.append(self.add_branch(multi_factor_features))
        descriptors.append(self.all_branch(torch.cat([x[name] for name in self.grid_names], dim=1)))

        for name in self.satellite_names:
            descriptors.append(self.satellite_branches[name](x[name]))

        if self.his_branch is not None:
            descriptors.append(self.his_branch(x["his"]))

        return self.head(torch.cat(descriptors, dim=1))

    @property
    def required_inputs(self) -> List[str]:
        names = self.grid_names + self.satellite_names
        if self.use_his:
            names.append("his")
        return names


class RITCFModel(nn.Module):
    def __init__(
        self,
        field20_names: Sequence[str] = ("u", "v", "pv"),
        field5_names: Sequence[str] = ("sst",),
        satellite_names: Sequence[str] = ("sat",),
        use_his: bool = True,
        his_dim: int = 30,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.encoder_a = SampleEncoder(
            field20_names=field20_names,
            field5_names=field5_names,
            satellite_names=satellite_names,
            use_his=use_his,
            his_dim=his_dim,
            dropout=dropout,
        )
        self.encoder_b = SampleEncoder(
            field20_names=field20_names,
            field5_names=field5_names,
            satellite_names=satellite_names,
            use_his=use_his,
            his_dim=his_dim,
            dropout=dropout,
        )

    def forward(self, input_a: Mapping[str, Tensor], input_b: Mapping[str, Tensor]) -> Tensor:
        emb_a = self.encoder_a(input_a)
        emb_b = self.encoder_b(input_b)
        return torch.sqrt(torch.sum((emb_a - emb_b) ** 2, dim=1, keepdim=True) + 1.0e-8)

    @property
    def required_inputs(self) -> List[str]:
        return self.encoder_a.required_inputs


def build_model(
    field20_names: Sequence[str] = ("u", "v", "pv"),
    field5_names: Sequence[str] = ("sst",),
    satellite_names: Sequence[str] = ("sat",),
    use_his: bool = True,
    his_dim: int = 30,
    dropout: float = 0.0,
) -> RITCFModel:
    return RITCFModel(
        field20_names=field20_names,
        field5_names=field5_names,
        satellite_names=satellite_names,
        use_his=use_his,
        his_dim=his_dim,
        dropout=dropout,
    )


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def random_batch(model: RITCFModel, batch_size: int = 2, his_dim: int = 30) -> Dict[str, Tensor]:
    batch: Dict[str, Tensor] = {}
    for name in model.encoder_a.field20_names:
        batch[name] = torch.randn(batch_size, 20, 25, 25)
    for name in model.encoder_a.field5_names:
        batch[name] = torch.randn(batch_size, 5, 25, 25)
    for name in model.encoder_a.satellite_names:
        batch[name] = torch.randn(batch_size, 5, 224, 224)
    if model.encoder_a.use_his:
        batch["his"] = torch.randn(batch_size, his_dim)
    return batch


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--field20", nargs="*", default=["u", "v", "pv"])
    parser.add_argument("--field5", nargs="*", default=["sst"])
    parser.add_argument("--satellite", nargs="*", default=["sat"])
    parser.add_argument("--no-his", action="store_true")
    parser.add_argument("--his-dim", type=int, default=30)
    parser.add_argument("--dropout", type=float, default=0.0)
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    model = build_model(
        field20_names=args.field20,
        field5_names=args.field5,
        satellite_names=args.satellite,
        use_his=not args.no_his,
        his_dim=args.his_dim,
        dropout=args.dropout,
    )
    print(model)
    print(f"required inputs: {model.required_inputs}")
    print(f"descriptor dim: {model.encoder_a.descriptor_dim}")
    print(f"encoder output dim: {model.encoder_a.output_dim}")
    print(f"trainable parameters: {count_parameters(model):,}")

    input_a = random_batch(model, batch_size=2, his_dim=args.his_dim)
    input_b = random_batch(model, batch_size=2, his_dim=args.his_dim)
    with torch.no_grad():
        output = model(input_a, input_b)
    print(f"test output shape: {tuple(output.shape)}")


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'torch'

In [4]:
from __future__ import annotations

import argparse
from pathlib import Path
from typing import Dict, Mapping, Optional, Sequence, Tuple

import numpy as np
import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset

try:
    from .model import RITCFModel, build_model
except ImportError:
    from model import RITCFModel, build_model

"""
example:

python train.py \
  --data-dir .\data \
  --save-path .\checkpoints\best_model.pt \
  --field20 u v pv \
  --field5 sst \
  --satellite sat \
  --his-dim 30 \
  --epochs 60 \
  --batch-size 256 \
  --lr 0.0001

data file named:
    train_u_A.npy
    train_u_B.npy
    train_v_A.npy
    train_v_B.npy
    train_pv_A.npy
    train_pv_B.npy
    ...

"""

ArrayDict = Dict[str, np.ndarray]
TensorDict = Dict[str, Tensor]


class PairDataset(Dataset):
    def __init__(self, input_a: ArrayDict, input_b: ArrayDict, label: np.ndarray) -> None:
        self.input_a = input_a
        self.input_b = input_b
        self.label = label.astype(np.float32).reshape(-1, 1)
        n = len(self.label)
        for group_name, group in (("input_a", input_a), ("input_b", input_b)):
            for key, value in group.items():
                if len(value) != n:
                    raise ValueError(f"{group_name}[{key}] length={len(value)}, label length={n}")

    def __len__(self) -> int:
        return len(self.label)

    def __getitem__(self, idx: int) -> Tuple[TensorDict, TensorDict, Tensor]:
        a = {key: torch.from_numpy(value[idx]).float() for key, value in self.input_a.items()}
        b = {key: torch.from_numpy(value[idx]).float() for key, value in self.input_b.items()}
        y = torch.from_numpy(self.label[idx]).float()
        return a, b, y


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--data-dir", default=".", help="directory containing split_factor_A/B.npy files")
    parser.add_argument("--save-path", default="checkpoints/best_model.pt")
    parser.add_argument("--field20", nargs="*", default=["u", "v", "pv"], help="20x25x25 factors")
    parser.add_argument("--field5", nargs="*", default=["sst"], help="5x25x25 factors")
    parser.add_argument("--satellite", nargs="*", default=["sat"], help="5x224x224 factors")
    parser.add_argument("--no-his", action="store_true")
    parser.add_argument("--his-dim", type=int, default=30)
    parser.add_argument("--epochs", type=int, default=200)
    parser.add_argument("--batch-size", type=int, default=256)
    parser.add_argument("--lr", type=float, default=1.0e-4)
    parser.add_argument("--weight-decay", type=float, default=0.0)
    parser.add_argument("--dropout", type=float, default=0.0)
    parser.add_argument("--threshold", type=float, default=0.5)
    parser.add_argument("--num-workers", type=int, default=0)
    parser.add_argument("--seed", type=int, default=2025)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    return parser.parse_args()


def factor_names(args: argparse.Namespace) -> Sequence[str]:
    names = list(args.field20) + list(args.field5) + list(args.satellite)
    if not args.no_his:
        names.append("his")
    return names


def find_npy(data_dir: Path, split: str, name: str, side: Optional[str] = None) -> Path:
    side_tokens = {
        "A": ["A", "a", "inputA", "input_A", "InputA", "Input_A"],
        "B": ["B", "b", "inputB", "input_B", "InputB", "Input_B"],
        None: [""],
    }[side]

    candidates = []
    for token in side_tokens:
        if token:
            candidates.extend(
                [
                    data_dir / f"{split}_{name}_{token}.npy",
                    data_dir / f"{split}_{token}_{name}.npy",
                    data_dir / split / f"{name}_{token}.npy",
                    data_dir / split / f"{token}_{name}.npy",
                ]
            )
        else:
            candidates.extend(
                [
                    data_dir / f"{split}_{name}.npy",
                    data_dir / split / f"{name}.npy",
                ]
            )
    for path in candidates:
        if path.exists():
            return path
    joined = ", ".join(str(path) for path in candidates[:4])
    raise FileNotFoundError(f"cannot find {split} {name} {side or ''} npy, tried: {joined} ...")


def channel_first(array: np.ndarray, channels: int) -> np.ndarray:
    if array.ndim == 5:
        if array.shape[2] == channels:
            return array
        if array.shape[-1] == channels:
            return np.transpose(array, (0, 1, 4, 2, 3))
        return array
    if array.ndim < 4:
        return array
    if array.shape[1] == channels:
        return array
    if array.shape[-1] == channels:
        axes = [0, array.ndim - 1] + list(range(1, array.ndim - 1))
        return np.transpose(array, axes)
    return array


def load_split(data_dir: Path, split: str, args: argparse.Namespace) -> Tuple[ArrayDict, ArrayDict, np.ndarray]:
    input_a: ArrayDict = {}
    input_b: ArrayDict = {}
    for name in args.field20:
        input_a[name] = channel_first(np.load(find_npy(data_dir, split, name, "A")), 20).astype(np.float32)
        input_b[name] = channel_first(np.load(find_npy(data_dir, split, name, "B")), 20).astype(np.float32)
    for name in args.field5:
        input_a[name] = channel_first(np.load(find_npy(data_dir, split, name, "A")), 5).astype(np.float32)
        input_b[name] = channel_first(np.load(find_npy(data_dir, split, name, "B")), 5).astype(np.float32)
    for name in args.satellite:
        input_a[name] = channel_first(np.load(find_npy(data_dir, split, name, "A")), 5).astype(np.float32)
        input_b[name] = channel_first(np.load(find_npy(data_dir, split, name, "B")), 5).astype(np.float32)
    if not args.no_his:
        input_a["his"] = np.load(find_npy(data_dir, split, "his", "A")).astype(np.float32)
        input_b["his"] = np.load(find_npy(data_dir, split, "his", "B")).astype(np.float32)

    try:
        label_path = find_npy(data_dir, split, "label", None)
    except FileNotFoundError:
        label_path = find_npy(data_dir, split, "y", None)
    label = np.load(label_path).astype(np.float32).reshape(-1)
    return input_a, input_b, label


def normalization_stats(*splits: Tuple[ArrayDict, ArrayDict, np.ndarray]) -> Dict[str, Tuple[float, float]]:
    stats: Dict[str, Tuple[float, float]] = {}
    keys = splits[0][0].keys()
    for key in keys:
        arrays = []
        for input_a, input_b, _ in splits:
            arrays.append(input_a[key])
            arrays.append(input_b[key])
        min_value = min(float(np.nanmin(x)) for x in arrays)
        max_value = max(float(np.nanmax(x)) for x in arrays)
        stats[key] = (min_value, max_value)
    return stats


def apply_normalization(split: Tuple[ArrayDict, ArrayDict, np.ndarray], stats: Mapping[str, Tuple[float, float]]) -> None:
    input_a, input_b, _ = split
    for key, (min_value, max_value) in stats.items():
        denom = max(max_value - min_value, 1.0e-6)
        input_a[key] = ((input_a[key] - min_value) / denom).astype(np.float32)
        input_b[key] = ((input_b[key] - min_value) / denom).astype(np.float32)


def to_device(batch: Mapping[str, Tensor], device: torch.device) -> TensorDict:
    return {key: value.to(device, non_blocking=True) for key, value in batch.items()}


def confusion_from_distance(distance: np.ndarray, label: np.ndarray, threshold: float) -> Tuple[int, int, int, int]:
    true_ri = label.reshape(-1) < 0.5
    pred_ri = distance.reshape(-1) < threshold
    hit = int(np.sum(true_ri & pred_ri))
    false_alarm = int(np.sum(~true_ri & pred_ri))
    miss = int(np.sum(true_ri & ~pred_ri))
    correct_negative = int(np.sum(~true_ri & ~pred_ri))
    return hit, false_alarm, miss, correct_negative


def metrics_from_counts(hit: int, false_alarm: int, miss: int, correct_negative: int) -> Dict[str, float]:
    pod = hit / (hit + miss) if (hit + miss) else 0.0
    farate = false_alarm / (false_alarm + correct_negative) if (false_alarm + correct_negative) else 0.0
    return {"POD": pod, "FARate": farate}


def run_epoch(
    model: RITCFModel,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    threshold: float,
    optimizer: Optional[torch.optim.Optimizer] = None,
) -> Dict[str, float]:
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    total_count = 0
    all_distance = []
    all_label = []

    for input_a, input_b, label in loader:
        input_a = to_device(input_a, device)
        input_b = to_device(input_b, device)
        label = label.to(device)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            distance = model(input_a, input_b)
            loss = criterion(distance, label)
            if is_train:
                loss.backward()
                optimizer.step()

        batch_size = len(label)
        total_loss += float(loss.detach().cpu()) * batch_size
        total_count += batch_size
        all_distance.append(distance.detach().cpu().numpy())
        all_label.append(label.detach().cpu().numpy())

    distance_np = np.concatenate(all_distance, axis=0)
    label_np = np.concatenate(all_label, axis=0)
    hit, false_alarm, miss, correct_negative = confusion_from_distance(distance_np, label_np, threshold)
    metrics = metrics_from_counts(hit, false_alarm, miss, correct_negative)
    metrics["loss"] = total_loss / max(total_count, 1)
    return metrics


def main() -> None:
    args = parse_args()
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)

    data_dir = Path(args.data_dir)
    train_split = load_split(data_dir, "train", args)
    val_split = load_split(data_dir, "val", args)
    test_split = load_split(data_dir, "test", args)

    stats = normalization_stats(train_split)
    apply_normalization(train_split, stats)
    apply_normalization(val_split, stats)
    apply_normalization(test_split, stats)

    train_loader = DataLoader(
        PairDataset(*train_split),
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=args.device.startswith("cuda"),
    )
    val_loader = DataLoader(
        PairDataset(*val_split),
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=args.device.startswith("cuda"),
    )

    device = torch.device(args.device)
    model = build_model(
        field20_names=args.field20,
        field5_names=args.field5,
        satellite_names=args.satellite,
        use_his=not args.no_his,
        his_dim=args.his_dim,
        dropout=args.dropout,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    criterion = nn.MSELoss()

    save_path = Path(args.save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    best_val_loss = float("inf")

    for epoch in range(1, args.epochs + 1):
        train_metrics = run_epoch(model, train_loader, criterion, device, args.threshold, optimizer=optimizer)
        val_metrics = run_epoch(model, val_loader, criterion, device, args.threshold)

        print(
            f"Epoch {epoch:03d} "
            f"train_loss={train_metrics['loss']:.6f} "
            f"train_POD={train_metrics['POD']:.4f} "
            f"train_FARate={train_metrics['FARate']:.4f} "
            f"val_loss={val_metrics['loss']:.6f} "
            f"val_POD={val_metrics['POD']:.4f} "
            f"val_FARate={val_metrics['FARate']:.4f}"
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            torch.save(
                {
                    "model": model.state_dict(),
                    "field20": list(args.field20),
                    "field5": list(args.field5),
                    "satellite": list(args.satellite),
                    "use_his": not args.no_his,
                    "his_dim": args.his_dim,
                    "threshold": args.threshold,
                    "normalization": stats,
                    "best_val_loss": best_val_loss,
                },
                save_path,
            )
            print(f"Saved best model to {save_path}")


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'torch'